# Get to Know a Dataset: Active Visual Semantics (AVS)

This notebook is a guided tour of the [Active Visual Semantics (AVS)](https://registry.opendata.aws/avs/) dataset — a multimodal neuroimaging dataset combining magnetoencephalography (MEG), eye-tracking, and structural MRI, recorded from 5 participants freely viewing natural scenes from MS-COCO. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

Everything below runs against the live, public `s3://kietzmannlab-avs` bucket — no AWS account or credentials needed (anonymous/unsigned S3 access).

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

At the top level of our S3 bucket, `kietzmannlab-avs`, there are four key prefixes:

1. **`sub-XX/`** — raw, unprocessed data per subject and session (`ses-XX`), split into `meg/` (native FIF-format MEG recordings), `eyetrack/` (native EDF eye-tracking recordings plus parsed event/sample tables), and `beh/` (behavioural/experiment-log data from the scene-viewing and caption-generation task).
2. **`derivatives/pyavs/`** — preprocessed derivatives, mirroring the same `sub-XX/ses-XX/` structure: Maxwell-filtered MEG, ICA solutions, cleaned eye-tracking events, and precomputed fixation-/saccade-locked MEG epochs.
3. **`stimuli/`** — metadata about the MS-COCO scene images shown to participants (object/category annotations, licensing information, scene-sampling tables). Note there are no scene image files themselves in this prefix — see the note on image redistribution below.
4. **`catalog/`** — small Parquet index files that support fast, content-indexed queries (e.g. "every fixation on a dog, across subjects") without downloading whole session files; used internally by the companion `pyavs` Python package.

The full dataset covers 5 subjects (`sub-01`–`sub-05`) x up to 10 sessions each. **As of this writing, the bucket holds a partial release** (`sub-01`, sessions `ses-01`–`ses-04`, ~54 GB) while the remaining bulk upload is in progress — the cell below reflects exactly what's live right now.

Let's look at this directly.

First, we'll import the libraries used throughout this notebook.

In [ ]:
# This notebook requires the following additional libraries
# (install with e.g. pip):
#
# boto3 >= 1.38
# pandas >= 2.0
# pyarrow >= 14.0
# matplotlib >= 3.8
# seaborn >= 0.13

import io

import boto3
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from botocore import UNSIGNED
from botocore.config import Config

sns.set_context("poster")

BUCKET = "kietzmannlab-avs"

# This is a public bucket, so requests don't need to be signed.
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED), region_name="us-west-2")

Now let's list the top-level prefixes in the bucket.

In [ ]:
for item in s3.list_objects_v2(Bucket=BUCKET, Delimiter="/")["CommonPrefixes"]:
    print(item["Prefix"])

As expected: `catalog/`, `derivatives/`, `stimuli/`, and `sub-01/`. Looking one level deeper into `sub-01/`, we find the 4 currently-public sessions, each with the `beh/`/`eyetrack/`/`meg/` structure described above.

In [ ]:
print("Sessions for sub-01:")
for item in s3.list_objects_v2(Bucket=BUCKET, Prefix="sub-01/", Delimiter="/")["CommonPrefixes"]:
    print(" ", item["Prefix"])

print("\nContents of sub-01/ses-01/:")
for item in s3.list_objects_v2(Bucket=BUCKET, Prefix="sub-01/ses-01/", Delimiter="/")["CommonPrefixes"]:
    print(" ", item["Prefix"])

print("\nRaw MEG recordings for sub-01/ses-01/meg/:")
for item in s3.list_objects_v2(Bucket=BUCKET, Prefix="sub-01/ses-01/meg/")["Contents"]:
    print(f"  {item['Key']}  ({item['Size'] / 1e6:.1f} MB)")

And the matching preprocessed derivatives for the same subject/session, under `derivatives/pyavs/`:

In [ ]:
print("Contents of derivatives/pyavs/sub-01/ses-01/:")
for item in s3.list_objects_v2(Bucket=BUCKET, Prefix="derivatives/pyavs/sub-01/ses-01/", Delimiter="/")["CommonPrefixes"]:
    print(" ", item["Prefix"])

print("\nPrecomputed epochs:")
for item in s3.list_objects_v2(Bucket=BUCKET, Prefix="derivatives/pyavs/sub-01/ses-01/epochs/")["Contents"]:
    print(f"  {item['Key']}  ({item['Size'] / 1e6:.1f} MB)")

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

Four formats cover the whole dataset:

- **FIF** (`.fif`) — raw and preprocessed MEG recordings, in the native format of the Neuromag/MEGIN MEG system and the [MNE-Python](https://mne.tools/) ecosystem. FIF is a binary, self-describing format that bundles sensor geometry, digitization, and channel metadata alongside the continuous recording, which is why it's the standard exchange format for MEG/EEG. Load with `mne.io.read_raw_fif`.
- **EDF** (`.EDF`) — raw eye-tracking recordings, in EyeLink's native binary format. We also ship the EyeLink-parsed events/samples/messages as Parquet directly (see below), so most users won't need to touch the EDF files at all unless they want to re-run EyeLink's own parser.
- **Parquet** — every tabular file in the dataset: parsed eye-tracking events/samples, behavioural/experiment logs, epoch metadata, and scene metadata. We migrated these from the lab's original CSV files to Parquet for this release — Parquet's columnar, compressed storage is both smaller on disk and much faster to partially read (e.g. only the columns you need) than CSV, and it's natively supported by `pandas`/`polars`/`pyarrow` as well as query engines like [Amazon Athena](https://docs.aws.amazon.com/athena/latest/ug/querying-parquet-data.html), which can query these files directly from S3 with SQL without downloading them.
- **HDF5** (`.h5`) — precomputed, per-epoch MEG data (fixation-/saccade-locked). Chunked one epoch at a time so that content-indexed queries (via the `catalog/` index and `pyavs`'s remote client) can range-read just the matching epochs instead of downloading a whole session's worth of data.

**A note on what's *not* included:** the MS-COCO scene images participants viewed are not redistributed as image files in this bucket — COCO/Flickr source photos don't carry a redistribution license we can ship under. Instead, `stimuli/` ships a small lookup table mapping each scene to its original COCO URL and license, and the `pyavs` package (see "Where to go next" below) reconstructs each scene image on demand from COCO's own hosting.

### Q: Can you show us an example of downloading and loading data from your dataset?

As an example, let's load the experiment log for `sub-01`, `ses-01` — one row per trial, recording which scene was shown, in which block, whether that trial included the caption-generation task, and so on.

In [ ]:
key = "sub-01/ses-01/beh/as_exp_data_1_1_3_0.parquet"

buf = io.BytesIO()
s3.download_fileobj(BUCKET, key, buf)
buf.seek(0)
explog = pd.read_parquet(buf)

print(explog.shape)
explog.head()

300 rows — one per trial in this session. The columns worth calling out: `scene_ID`/`scene_filename` identify the MS-COCO scene shown; `scene_duration` and `ITI` give the trial timing; `caption_task` flags whether this trial also asked the participant to describe the scene out loud (`captions` holds the transcription); and the `meas_time_*` columns record measured (as opposed to nominally-requested) durations of each trial phase, useful for verifying timing precision.

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

First, where on the screen did the participant actually look? We'll load the preprocessed eye-tracking events for the same subject/session and plot the screen-coordinate density of all fixations.

In [ ]:
key = "derivatives/pyavs/sub-01/ses-01/eyetrack/as_s1_el_events.parquet"

buf = io.BytesIO()
s3.download_fileobj(BUCKET, key, buf)
buf.seek(0)
events = pd.read_parquet(buf)

fixations = events[events["type"] == "fixation"].copy()

# This file covers the whole recording block, not just scene-viewing time, so
# a small number of "fixation"-coded events (~4%) are multi-second stretches
# recorded during inter-trial breaks/calibration rather than genuine gaze
# fixations. We drop those here using a standard perceptual-fixation cutoff;
# Section 5 below shows how pyavs's own preprocessing does this filtering
# properly, by aligning events to trial boundaries instead of a fixed cutoff.
fixations = fixations[fixations["duration"] <= 2.0]
print(f"{len(fixations)} fixations")

plt.figure(figsize=(8, 6))
sns.kdeplot(data=fixations, x="mean_gx", y="mean_gy", fill=True, cmap="magma", thresh=0.02)
plt.gca().invert_yaxis()  # screen y grows downward
plt.xlabel("gaze x [screen px]")
plt.ylabel("gaze y [screen px]")
sns.despine()
plt.tight_layout()
plt.show()

A classic center bias, as expected for free viewing of centrally-composed natural scenes. Next, how long do fixations last?

In [ ]:
fixations["duration_ms"] = fixations["duration"] * 1000

plt.figure(figsize=(8, 6))
sns.histplot(data=fixations, x="duration_ms", color="cornflowerblue", bins=40)
plt.xlabel("fixation duration [ms]")
plt.ylabel("count [fixations]")
sns.despine()
plt.tight_layout()
plt.show()

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**Does fixation duration vary with *what* is being looked at?** AVS's precomputed epoch metadata carries a COCO-Stuff object-category label for every scene fixation, letting us ask this directly. But first, getting a clean per-trial view of the eye-tracking stream — ordering fixations within each trial and separating scene-viewing from inter-trial-interval events — requires aligning the raw event stream to trial boundaries from the experiment log, exactly the kind of preprocessing the companion [`pyavs`](https://github.com/KietzmannLab/pyavs) Python package handles. `pyavs.open_remote()` gives it the same S3-backed access we've been using directly above, so nothing needs to be downloaded or staged by hand.

In [ ]:
# pip install git+https://github.com/KietzmannLab/pyavs.git
import pyavs

avs = pyavs.open_remote()
avs.load_experiment_log(subject_id=1, session=1)
avs.load_eye_events(subject_id=1, session=1)

_, enriched_events = pyavs.load_and_enrich_eye_events(subjects=[1], sessions=[1], data_path=avs.data_path)

scene_fixations = enriched_events[
    (enriched_events["type"] == "fixation") & (enriched_events["recording"] == "scene")
].copy()
scene_fixations["duration_ms"] = scene_fixations["duration"] * 1000

print(f"{len(scene_fixations)} scene-viewing fixations")

Before turning to how fixation duration relates to what's being looked at, it's worth seeing what a single trial actually looks like. Below is a trial roughly in the middle of this session — the scanpath (fixations, in presentation order) and the underlying raw eye-tracking samples, both plotted directly on the scene image. AVS doesn't ship the scene images themselves in the release (to avoid redistributing copyrighted photos without per-image attribution); `pyavs` fetches them on demand from COCO's own hosting and reproduces the AVS experiment's exact crop/resize, caching the result locally.

In [ ]:
# Scene images aren't in the release bucket -- pyavs fetches them from COCO on demand.
# That needs a small (~150 KB) per-image license/URL table, fetched once here.
import os
from PIL import Image
from pyavs.layout import get_layout
from pyavs.config.config import PyAVSConfig

os.makedirs(f"{avs.data_path}/stimuli", exist_ok=True)
s3.download_file(BUCKET, "stimuli/avs_scenes_all_licenses.parquet",
                  f"{avs.data_path}/stimuli/avs_scenes_all_licenses.parquet")

layout = get_layout(avs.data_path)
config = PyAVSConfig()
screen_w, screen_h = config.screen_size_pixels

trial_id = int(scene_fixations["trial"].mean()) # pick a trial in the middle of the session
trial_fixations = scene_fixations[scene_fixations["trial"] == trial_id].sort_values("fix_sequence")
scene_id = int(trial_fixations["sceneID"].iloc[0])

scene_img = Image.open(layout.ensure_scene_image(scene_id))
rescaled_size = config.get_rescaled_scene_size(scene_img.size)
if rescaled_size != scene_img.size:
    scene_img = scene_img.resize(rescaled_size)
img_w, img_h = rescaled_size

print(f"trial {trial_id}, scene {scene_id}: {len(trial_fixations)} fixations")

In [ ]:
# Gaze coordinates have (0, 0) at the screen's top-left (EyeLink convention) and grow
# downward; the scene is displayed centered on the screen. To overlay correctly we
# re-center both onto the same axes.
x = trial_fixations["mean_gx"] - screen_w / 2
y =  trial_fixations["mean_gy"] - screen_h / 2 

plt.figure(figsize=(8, 6))
ax = plt.gca()
ax.imshow(scene_img, extent=[-img_w / 2, img_w / 2, -img_h / 2, img_h / 2])
ax.plot(x, y, color="white", alpha=0.8)
sc = ax.scatter(x, y, c=trial_fixations["fix_sequence"], cmap="magma", s=200,
                 edgecolors="white", linewidth=1.5)
cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("fixation order [index]")
ax.axis("off")
plt.tight_layout()
plt.show()

And the same trial's raw gaze samples (sampled around 500 Hz), each tagged `fixation`/`saccade`/`blink` by the EyeLink preprocessing rather than reduced to discrete events — the fixation clusters below show individual dwell time within each fixation, and the saccade samples in between retrace the same scanpath as the events plot above.

In [ ]:
key = "derivatives/pyavs/sub-01/ses-01/eyetrack/as_s1_el_samples.parquet"

buf = io.BytesIO()
s3.download_fileobj(BUCKET, key, buf)
buf.seek(0)
samples = pd.read_parquet(buf)

t0 = trial_fixations["start_time"].min()
t1 = trial_fixations["end_time"].max()
trial_samples = samples[(samples["smpl_time"] >= t0) & (samples["smpl_time"] <= t1)].copy()

x = trial_samples["gx"] - screen_w / 2
y = trial_samples["gy"] - screen_h / 2


plt.figure(figsize=(8, 6))
ax = plt.gca()
ax.imshow(scene_img, extent=[-img_w / 2, img_w / 2, -img_h / 2, img_h / 2])
sns.scatterplot(x=x, y=y, hue=trial_samples["type"], hue_order=["fixation", "saccade", "blink"],
                 palette=["cornflowerblue", "salmon", "black"], s=40, edgecolor="none",
                 alpha=0.9, ax=ax)
ax.legend(frameon=False, loc="upper right")
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"{len(trial_samples)} samples over {t1 - t0:.1f} s "
      f"({(trial_samples['type'] == 'fixation').sum()} fixation, "
      f"{(trial_samples['type'] == 'saccade').sum()} saccade)")

Now, back to the question: does fixation duration vary with *what* is being looked at? AVS's precomputed epoch metadata already carries a COCO-Stuff object-category label for each scene fixation — the same per-fixation metadata `pyavs`'s own `scripts/et_viz/plot_fixation_events_with_objects.py` uses for its object-fixation summaries. We reuse it here for the most-fixated categories in this session.

In [ ]:
key = "derivatives/pyavs/sub-01/ses-04/epochs/sub-01_ses-04_fixation_metadata.parquet"

buf = io.BytesIO()
s3.download_fileobj(BUCKET, key, buf)
buf.seek(0)
fix_meta = pd.read_parquet(buf)

# 'None' / 'outside' / 'unknown' aren't real object categories -- drop them, as pyavs's
# own object-fixation summary scripts do.
# filter for scene fixations and drop the most extreme durations top 2% 
fix_meta = fix_meta[fix_meta["recording"] == "scene"].copy()
fix_meta = fix_meta[fix_meta["duration"] <= fix_meta["duration"].quantile(0.98)].copy()
fix_meta = fix_meta[~fix_meta["object_label"].isin(["None", "outside", "unknown"])].copy()
fix_meta["duration_ms"] = fix_meta["duration"] * 1000

top_objects = fix_meta["object_label"].value_counts().head(15).index.tolist()
# sort descending by median fixation duration, so the barplot is easier to read
top_objects = fix_meta.groupby("object_label")["duration_ms"].median().loc[top_objects].sort_values(ascending=False).index.tolist()

by_object = fix_meta[fix_meta["object_label"].isin(top_objects)]

plt.figure(figsize=(10, 6))
sns.barplot(data=by_object, x="object_label", y="duration_ms", order=top_objects,
            estimator="median", errorbar=("ci", 95), palette="husl")
plt.xticks(rotation=45, ha="right")
plt.xlabel("fixated object category")
plt.ylabel("median fixation duration [ms]")
sns.despine()
plt.tight_layout()
plt.show()

Across these fifteen categories, median fixation duration ranges from about 230 ms (wall-concrete) to 303 ms (cat) — a real but modest ~73 ms spread, with overlapping 95% confidence intervals for most pairs. (Single-subject, single-session again — not a claim about object semantics in general; a dataset-wide version of this, pooling fixations across all 5 subjects via `pyavs.remote`'s content-indexed epoch queries, would be a natural follow-up.)

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**Does the magnitude of the fixation-locked early visual response (an M100-like deflection) scale with fixation duration or incoming saccade kinematics (amplitude, peak velocity)?** The dataset paper establishes a robust fixation-locked visual response peaking around 114 ms post-fixation onset — both in sensor-space single-fixation encoding and in source-localised, category-averaged evoked responses — but doesn't ask whether the *size* of that response depends on the eye movement that produced the fixation. AVS's fixation-locked epochs carry per-fixation eye-movement kinematics directly alongside the MEG response, so this doesn't require any new preprocessing.

A reasonable starting point: `pyavs`'s fixation-aligned analysis pipelines — the same machinery behind the dataset paper's fixation-locked representational similarity analysis (`scripts/rsa_analysis/`) and source-projected fixation ERFs (`scripts/meg_viz/compute_source_erp.py`) — already extract per-fixation, response-locked amplitude time courses at both sensor and source level. Regressing peak amplitude (e.g. at the ~114 ms peak latency reported in the paper) against fixation duration and saccade kinematics, with scene and subject as random effects, would extend that pipeline directly to this question, and scales naturally across subjects/sessions via `pyavs.remote`'s content-indexed epoch queries.

## Where to go next

This notebook only scratched the surface — raw and preprocessed data, no analysis pipeline. For more:

- **pyAVS Colab quickstart** — a deeper, `pyavs`-native walkthrough (raw MEG + eye-tracking, scene-image reconstruction, a sensor-space MEG contrast) that runs end-to-end in your browser, no local setup: [`pyavs_colab_quickstart.ipynb`](https://colab.research.google.com/github/KietzmannLab/pyavs/blob/main/examples/pyavs_colab_quickstart.ipynb)
- **Full dataset documentation** (structure, known issues, methods, citation): [kietzmannlab.uni-osnabrueck.de/avs/](https://kietzmannlab.uni-osnabrueck.de/avs/)
- **The `pyavs` package and its analysis pipelines** (source reconstruction, representational similarity analysis, ANN-to-MEG encoding, decoding): [github.com/KietzmannLab/pyavs](https://github.com/KietzmannLab/pyavs)

AVS is released under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) — see the [citation page](https://kietzmannlab.uni-osnabrueck.de/avs/reference/citation.html) for how to cite the dataset.